# Week 10, Day 1: Integrals: accumulating change, from rates to spike counts

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w010/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Three ways to integrate

Write `left(f, a, b, n)`, `midpoint(f, a, b, n)` and `trapezoid(f, a, b, n)`, each using n pieces. For $\int_0^1 e^t\,dt = e - 1$, store each rule's absolute errors for n = 1, 10 and 100 in the dictionary `errs`, and read off how fast each falls.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def left(f, a, b, n):
    t = np.linspace(a, b, n + 1)[:-1]
    return ...

def midpoint(f, a, b, n):
    dt = (b - a) / n
    t = a + dt * (np.arange(n) + 0.5)
    return ...

def trapezoid(f, a, b, n):
    t = np.linspace(a, b, n + 1)
    y = f(t)
    return ...

exact = np.e - 1
errs = {name: [abs(rule(np.exp, 0, 1, n) - exact) for n in [1, 10, 100]] for name, rule in [("left", left), ("midpoint", midpoint), ("trapezoid", trapezoid)]}
for k, v in errs.items():
    print(f"{k:9s}", np.array(v))

In [ ]:
import numpy as np
def test_exact():
    "each rule integrates a straight line from 0 to 1 correctly enough"
    f = lambda t: 2 * t + 1
    assert np.isclose(midpoint(f, 0, 1, 10), 2.0) and np.isclose(trapezoid(f, 0, 1, 10), 2.0)
    assert np.isclose(left(f, 0, 1, 1000), 2.0, atol=2e-3)

def test_orders():
    "multiplying n by 10 cuts the left error about 10 times and the midpoint and trapezoid errors about 100 times"
    assert 8 < errs["left"][1] / errs["left"][2] < 12
    assert 80 < errs["midpoint"][1] / errs["midpoint"][2] < 120
    assert 80 < errs["trapezoid"][1] / errs["trapezoid"][2] < 120

test_exact()
test_orders()
print("All checks passed")

### Exercise: Integrate, then simulate

The receptor's rate is 5 Hz before the stimulus at 0.2 s, then $5 + 60e^{-(t - 0.2)/0.15} + 20(1 - e^{-(t - 0.2)/0.05})$. Compute `expected_count` over 0 to 1 s with the trapezoid rule, a `closed_form` answer using $\int_0^T e^{-s/\tau}ds = \tau(1 - e^{-T/\tau})$, and the mean count over 2000 simulated trials in 1 ms bins (`mean_simulated`).

In [ ]:
def receptor_rate(t):
    t = np.asarray(t, dtype=float)
    s = np.clip(t - 0.2, 0, None)
    return np.where(t < 0.2, 5.0, 5 + 60 * np.exp(-s / 0.15) + 20 * (1 - np.exp(-s / 0.05)))

expected_count = ...

T = 0.8                                     # the stimulus lasts from 0.2 s to 1 s
closed_form = 5 * 1.0 + ... + ...           # baseline for the whole second, plus the two stimulus terms

rng = np.random.default_rng(1926)
dt = 0.001
t = np.arange(0, 1, dt)
spikes = rng.random((2000, len(t))) < receptor_rate(t) * dt
mean_simulated = ...
print(f"integral {expected_count:.3f}, closed form {closed_form:.3f}, simulated {mean_simulated:.3f} spikes")

In [ ]:
import numpy as np
def test_integral():
    "expected_count is the integral of the rate over one second"
    assert np.isclose(expected_count, trapezoid(receptor_rate, 0, 1, 100000), rtol=1e-6)

def test_closed():
    "the closed-form answer agrees with the numerical integral"
    assert np.isclose(closed_form, expected_count, rtol=1e-4)

def test_sim():
    "the simulated mean count is within 3% of the integral"
    assert abs(mean_simulated - expected_count) / expected_count < 0.03

test_integral()
test_closed()
test_sim()
print("All checks passed")

### Exercise: Cumulative sums and differences

With 1 ms bins, build `N_cum`, the running total of rate × dt (use `np.cumsum`). Recover the rate from it by differencing (`np.diff` divided by dt) into `rate_back`. Then use `N_cum` to get the expected count between 0.3 and 0.5 s without integrating again (`count_window`).

In [ ]:
N_cum = ...
rate_back = ...
count_window = N_cum[int(round(0.5 / dt)) - 1] - N_cum[int(round(0.3 / dt)) - 1]

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(t, receptor_rate(t)); ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("rate (Hz)")
ax[1].plot(t, N_cum); ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("expected spikes so far")
plt.tight_layout(); plt.show()
print("expected spikes between 0.3 and 0.5 s:", round(count_window, 3))

In [ ]:
import numpy as np
def test_cum():
    "N_cum[k] approximates the integral of the rate from 0 to t[k] + dt"
    assert np.isclose(N_cum[-1], np.sum(receptor_rate(t)) * dt)

def test_back():
    "differencing N_cum recovers the rate"
    assert np.allclose(rate_back, receptor_rate(t)[1:], rtol=1e-6)

def test_window():
    "count_window is N(0.5) - N(0.3)"
    assert np.isclose(count_window, N_cum[int(round(0.5 / dt)) - 1] - N_cum[int(round(0.3 / dt)) - 1])

test_cum()
test_back()
test_window()
print("All checks passed")